# 🎬 AI Movie Translate (v2.2) — Google Colab GPU Edition
### ⚡ 100% Full Movie Dialogue Translation & Dubbing Pipeline
**Autonomous Multi-Agent • Full Dialogue Translation • Colloquial Spoken Burmese • Free Cloud GPU**

[![GitHub](https://img.shields.io/badge/GitHub-paipai1999%2Fai--translate--agent-blue?logo=github)](https://github.com/paipai1999/ai-translate-agent)
[![Colab](https://img.shields.io/badge/Google%20Colab-T4%20GPU%20Ready-orange?logo=googlecolab)](https://colab.research.google.com)
[![License](https://img.shields.io/badge/License-MIT-green)](https://github.com/paipai1999/ai-translate-agent/blob/main/LICENSE)

---

### 📌 အရေးကြီးသော ကြိုတင်ပြင်ဆင်မှု (Pre-requisite):
ဤစနစ်သည် **NVIDIA GPU** ဖြင့် Run လျှင် အသံဖမ်းယူမှု (Whisper STT)၊ Demucs အသံခွဲထုတ်မှု နှင့် Video Rendering များ **၅ ဆ မှ ၁၀ ဆ ခန့် ပိုမိုမြန်ဆန်** ပါသည်။
1. မီးနူးဘားမှ: **Runtime** > **Change runtime type** ကို နှိပ်ပါ။
2. **Hardware accelerator** တွင် **`T4 GPU`** (အခမဲ့) ကို ရွေးပေးပါ။
3. **Save** ကို နှိပ်ပါ။


## 🔍 Pre-Flight: Check GPU & Hardware Acceleration

In [ ]:
# Check GPU availability
import torch
import subprocess

print("=== 🖥️ HARDWARE SPECIFICATIONS ===")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ Active GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"✅ PyTorch CUDA Version: {torch.version.cuda}")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
else:
    print("⚠️ WARNING: No GPU detected! Currently running on CPU.")
    print("👉 Please go to: Runtime > Change runtime type > Select 'T4 GPU' for maximum speed.")

## 📦 Step 1: Clone Repository & Setup Workspace
Google Drive ချိတ်ဆက်စရာမလိုဘဲ Colab ၏ High-Speed Local SSD ပေါ်တွင် တိုက်ရိုက် Clone လုပ်ပြီး Run ပါမည်။


In [ ]:
# 1. Clone repository directly into Colab local storage (No Google Drive needed)
import os

project_dir = "/content/ai-translate-agent"
if not os.path.exists(project_dir):
    print("[*] Cloning repository...")
    %cd /content
    !git clone https://github.com/paipai1999/ai-translate-agent.git
else:
    print("[*] Existing repository found, fetching latest updates...")
    %cd {project_dir}
    !git reset --hard HEAD
    !git pull origin main

%cd {project_dir}
os.makedirs("temp", exist_ok=True)
print("\n✅ Working Directory: " + os.getcwd())
print("⚡ Running purely on Colab High-Speed Local SSD (Zero Drive mount needed)")


## 🛠️ Step 2: Install System Dependencies & Myanmar Fonts
FFmpeg၊ Myanmar Fonts (Padauk/Noto) နှင့် Python Libraries များကို တင်သွင်းပါမည်။

In [ ]:
print("[*] Installing Linux packages (FFmpeg, Myanmar Padauk Fonts)...")
!apt-get update -qq
!apt-get install -y -qq ffmpeg fonts-sil-padauk fonts-noto-cjk fonts-noto-core > /dev/null 2>&1

print("[*] Installing Python libraries from requirements.txt...")
!pip install -q -r requirements.txt

print("\n✅ System & Python dependencies successfully installed!")

## ⚙️ Step 3: Interactive Configuration Form
အောက်ပါ Form တွင် သင့် Gemini API Key နှင့် Settings များကို ရွေးချယ်ပြီး Run ပေးပါ:

In [ ]:
# @title 🎬 Video & AI Pipeline Settings { run: "auto" }

# @markdown ### 🔑 1. Google Gemini API Key
# @markdown အခမဲ့ API Key ကို [Google AI Studio](https://aistudio.google.com/app/apikey) တွင် ရယူပါ:
GEMINI_API_KEY = "" # @param {type:"string"}

# @markdown ### 🗣️ 2. Language & Voiceover Engine
LANGUAGE = "burmese" # @param ["burmese", "english"]
VOICE_ENGINE = "edge_tts" # @param ["edge_tts", "f5_tts"]
MYANMAR_VOICE = "my-MM-ThihaNeural" # @param ["my-MM-ThihaNeural", "my-MM-NilarNeural"]
ENGLISH_VOICE = "en-US-GuyNeural" # @param ["en-US-GuyNeural", "en-US-JennyNeural"]

# @markdown ### 🛡️ 4. Branding & Watermark Settings
WATERMARK_ENABLED = True # @param {type:"boolean"}
WATERMARK_TEXT = "PAI AI Movie Translate" # @param {type:"string"}
WATERMARK_OPACITY = 0.4 # @param {type:"number"}

# @markdown ### 🤖 3. Model & Processing Settings
WHISPER_MODEL = "small" # @param ["base", "small", "medium", "large-v3"]
GEMINI_MODEL = "gemini-3.5-flash-lite" # @param ["gemini-3.5-flash-lite", "gemini-3.6-flash"]
ANTI_COPYRIGHT = True # @param {type:"boolean"}
SUBTITLE_BLUR = True # @param {type:"boolean"}

import json
import os

if not GEMINI_API_KEY.strip():
    print("⚠️ သတိပေးချက်: GEMINI_API_KEY မထည့်ရသေးပါ။ https://aistudio.google.com/app/apikey မှ Key ထည့်ပေးပါ။")

config_data = {
    "gemini": {
        "enabled": True,
        "api_keys": [GEMINI_API_KEY.strip()] if GEMINI_API_KEY.strip() else [],
        "model": GEMINI_MODEL,
        "daily_limit_per_key": 20,
        "model_limits": {
            "gemini-3.5-flash-lite": 20,
            "gemini-3.6-flash": 15,
            "gemini-2.5-flash": 10,
            "gemini-2.0-flash": 15
        },
        "models": {
            "heavy": GEMINI_MODEL,
            "workhorse": GEMINI_MODEL,
            "polish": GEMINI_MODEL
        }
    },
    "pipeline": {
        "language": LANGUAGE,
        "whisper_model": WHISPER_MODEL,
        "scene_threshold": 30.0,
        "max_characters": 6,
        "max_scenes_for_llm": 30,
        "parallel_processing": True,
        "use_demucs": True
    },
    "voice": {
        "enabled": True,
        "engine": VOICE_ENGINE,
        "tts_voice_mm": MYANMAR_VOICE,
        "tts_voice_en": ENGLISH_VOICE,
        "tts_voice": MYANMAR_VOICE if LANGUAGE == "burmese" else ENGLISH_VOICE,
        "tts_rate_mm": "+8%",
        "tts_rate_en": "+15%",
        "f5_tts": {
            "model_type": "F5-TTS",
            "auto_character_cloning": True,
            "default_ref_audio": "assets/voices/default_ref.wav",
            "default_ref_text": "",
            "speed": 1.0,
            "device": "cuda" if torch.cuda.is_available() else "cpu"
        }
    },
    "batch": {
        "movies_folder": "movies",
        "max_parallel_jobs": 1,
        "skip_completed": True
    },
    "paths": {
        "temp_dir": "temp",
        "output_dir": "outputs"
    },
    "watermark": {
        "enabled": WATERMARK_ENABLED,
        "text": WATERMARK_TEXT,
        "opacity": WATERMARK_OPACITY,
        "margin": 30,
        "font_size": 40
    },
    "copyright_protection": {
        "enabled": ANTI_COPYRIGHT,
        "mirror_video": False,
        "resize_factor": 1.02
    },
    "subtitle_blur": {
        "enabled": SUBTITLE_BLUR,
        "region_height_pct": 0.18,
        "blur_strength": 18
    },
    "color_grading": {
        "enabled": True,
        "brightness": 0.03,
        "contrast": 1.02,
        "saturation": 1.08
    },
    "subtitle_overlay": {
        "enabled": True,
        "font_name": "Padauk",
        "font_size": 40,
        "bold": True,
        "border_style": 3,
        "outline_width": 3,
        "margin_bottom": 50,
        "max_chars_per_line": 28
    },
    "logging": {
        "level": "INFO",
        "save_log_file": True
    }
}

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=4)

print("\n✅ config.json created successfully!")
print(f"🎯 Language: {LANGUAGE.upper()} | Voice Engine: {VOICE_ENGINE.upper()} | LLM: {GEMINI_MODEL}")

## 🚀 Step 4 (Option A): 1-Click Video Execution (Command Line)
YouTube Link သို့မဟုတ် Video URL ကို ထည့်သွင်းပြီး တိုက်ရိုက် Run နိုင်ပါသည်:

In [ ]:
# @title 🎬 Run Autonomous Full Movie Dialogue Translation Pipeline
VIDEO_URL = "https://www.youtube.com/watch?v=5VRSIZwxJso" # @param {type:"string"}
CUSTOM_THUMBNAIL_TITLE = "" # @param {type:"string"}

import os

if not VIDEO_URL.strip():
    print("❌ ကျေးဇူးပြု၍ ဗီဒီယို Link ထည့်ပေးပါ။")
else:
    print(f"[*] Starting autonomous full dialogue translation for: {VIDEO_URL}")
    cmd = ["python", "main.py", VIDEO_URL, "-l", LANGUAGE, "-e", VOICE_ENGINE]
    if CUSTOM_THUMBNAIL_TITLE.strip():
        cmd.extend(["--thumb-title", f'"{CUSTOM_THUMBNAIL_TITLE.strip()}"'])
    if not WATERMARK_ENABLED:
        cmd.append("--no-watermark")
    elif WATERMARK_TEXT.strip():
        cmd.extend(["--watermark-text", f'"{WATERMARK_TEXT.strip()}"'])
    cmd_str = " ".join(cmd)
    !{cmd_str}


## 🌐 Step 4 (Option B): Launch Visual Web Dashboard (Interactive)
Browser ပေါ်တွင် Visual Interface ဖြင့် အသုံးပြုလိုပါက ဤ Cell ကို Run ပါ:

In [ ]:
# @title 🌐 Launch Web UI with Cloudflare Tunnel

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess, time

# Stop any existing server processes
!pkill -f "web_ui.py" || true
!pkill -f "cloudflared" || true

print("[*] Starting Web UI Background Server...")
server_proc = subprocess.Popen(
    ["python", "web_ui.py"],
    stdout=open("/content/web_ui.log", "w"),
    stderr=subprocess.STDOUT
)
time.sleep(4)

if server_proc.poll() is not None:
    print("❌ Web UI failed to start! Crash logs:")
    with open("/content/web_ui.log", "r") as f:
        print(f.read())
else:
    print("\n" + "="*60)
    print("🌐 CLOUDFLARE SECURE TUNNEL ACTIVE")
    print("👇 Click the link ending with '.trycloudflare.com' below 👇")
    print("="*60 + "\n")
    !./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:5000

## 🎬 Step 5: Output Player & Results Preview
ထွက်ရှိလာသော Recap Video၊ Subtitle၊ Script နှင့် Timing များကို Colab ပေါ်တွင် တိုက်ရိုက် ကြည့်ရှုပါ:

In [ ]:
import os
import json
from IPython.display import display, HTML, Video

outputs_dir = "outputs"
if not os.path.exists(outputs_dir) or not os.listdir(outputs_dir):
    print("ℹ️ No completed movies found yet in outputs/.")
else:
    for movie in sorted(os.listdir(outputs_dir)):
        movie_dir = os.path.join(outputs_dir, movie)
        if not os.path.isdir(movie_dir): continue
        
        final_mp4 = os.path.join(movie_dir, "final_recap.mp4")
        state_json = os.path.join(movie_dir, "state.json")
        script_txt = os.path.join(movie_dir, "final_recap_script.txt")
        
        print(f"\n{'='*60}")
        print(f"🎥 MOVIE: {movie}")
        print(f"{'='*60}")
        
        if os.path.exists(state_json):
            with open(state_json, 'r', encoding='utf-8') as f:
                s = json.load(f)
            total_dur = s.get("total_duration_formatted", "N/A")
            print(f"⏱️ Total Duration: {total_dur} | Phase: {s.get('current_phase')} | Progress: {s.get('progress')}%")
            print("⏱️ Timing Breakdown:")
            for phase, dur in s.get("phase_durations", {}).items():
                print(f"   • {phase:<40}: {dur}s")
                
        if os.path.exists(final_mp4):
            size_mb = os.path.getsize(final_mp4) / (1024 * 1024)
            print(f"\n✅ Final Recap Video Ready ({size_mb:.1f} MB): {final_mp4}")
            display(Video(final_mp4, embed=True, width=640))
            
        if os.path.exists(script_txt):
            print(f"\n📜 Script & SEO Preview (First 500 characters):")
            with open(script_txt, 'r', encoding='utf-8') as f:
                print(f.read()[:500] + "...")